Hypothesis testing & PRC of simple de novo simulation.

In [1]:
#imports
import scMPRAforge as scm
import pandas as pd
import numpy as np
from dask.distributed import Client, LocalCluster

2025-09-04 11:44:43.564583: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-04 11:44:43.568785: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

In [2]:
#set up cluster
cluster=LocalCluster(memory_limit='24GB')
client=Client(cluster)

In [4]:
#load sim & hypo test
data_root="/gpfs/gibbs/pi/reilly/tabula_data"
de_novo_sim=scm.de_novo_simulation.load(client,data_root,"simulated/simple_de_novo")
hypothesis_set=scm.HypothesisSet.from_tsv(f"{data_root}/simulated/simple_de_novo_hypotheses.tsv")

In [5]:
test_data=de_novo_sim.simulated_scMPRA[0].result()
test_data.ortho_filter()
primordial=scm.ortho()
primordial.criss_cross(client=client,
                       dat=test_data)
primordial.extract_params(client)
primordial.precompute_wald(client)

scMPRAforge: INFO: Dropped 0 of 150 (cell_type, cre_id) combos with fewer than 3 nonzero entries.


In [7]:
tester = scm.HypothesisTester("wald")
wald_results  = tester.run(hypothesis_set, primordial, client)

In [9]:
de_novo_sim.ground_truth

,cre_id,true_mean,cell_type
0,reference,1.000000,reference
1,CRE_even_1,23.111111,reference
2,CRE_even_2,45.222222,reference
3,CRE_even_3,67.333333,reference
4,CRE_even_4,89.444444,reference
...,...,...,...
5,CRE_even_5_high_in_neuron,112.555556,neuron
6,CRE_even_6_high_in_neuron,134.666667,neuron
7,CRE_even_7_high_in_neuron,156.777778,neuron
8,CRE_even_8_high_in_neuron,178.888889,neuron


In [10]:
results=wald_results.df
results

,comparison_CRE,reference_CRE,comparison_cell_type,reference_cell_type,meta,test_statistic,p_value,fold_change,flattened,test_type,bh_p
0,reference,CRE_even_1,reference,reference,<NA>,0.000000,1.000000,1.000000,False,wald,1.000000
1,reference,CRE_even_1,blood,blood,<NA>,0.000000,1.000000,1.000000,False,wald,1.000000
2,reference,CRE_even_1,neuron,neuron,<NA>,0.000000,1.000000,1.000000,False,wald,1.000000
3,reference,CRE_even_2,reference,reference,<NA>,0.000000,1.000000,1.000000,False,wald,1.000000
4,reference,CRE_even_2,blood,blood,<NA>,0.000000,1.000000,1.000000,False,wald,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
3820,CRE_even_5_high_in_neuron,CRE_even_5_high_in_neuron,blood,neuron,<NA>,0.305439,0.760032,1.009605,False,wald,0.813636
3821,CRE_even_6_high_in_neuron,CRE_even_6_high_in_neuron,blood,neuron,<NA>,1.097085,0.272604,1.080552,False,wald,0.293308
3822,CRE_even_7_high_in_neuron,CRE_even_7_high_in_neuron,blood,neuron,<NA>,0.675672,0.499249,1.030909,False,wald,0.535660
3823,CRE_even_8_high_in_neuron,CRE_even_8_high_in_neuron,blood,neuron,<NA>,0.698084,0.485125,1.032740,False,wald,0.520652


In [28]:
#merge in `comparison` ground truth
merged=results.merge(de_novo_sim.ground_truth,
    left_on=["comparison_CRE","comparison_cell_type"],
    right_on=["cre_id","cell_type"]
)
merged=merged.drop(columns=["cre_id","cell_type"])
merged=merged.rename({"true_mean":"comparison_truth"},axis=1)

#merge in `reference` ground truth
merged=merged.merge(de_novo_sim.ground_truth,
    left_on=["reference_CRE","reference_cell_type"],
    right_on=["cre_id","cell_type"]
)
merged=merged.drop(columns=["cre_id","cell_type"])
merged=merged.rename({"true_mean":"reference_truth"},axis=1)

#ground truth effect size
merged["gt_effect_size"]=merged["comparison_truth"]/merged["reference_truth"]
#ground truth null hypothesis that the CREs are the same : true or false?
merged["gt_null"]=abs(merged["gt_effect_size"]-1)<1e-8

merged["reject_null"]=merged["bh_p"]<0.05

pd.crosstab(merged["gt_null"],merged["reject_null"])

#merged
#simple_comparison_df=merged[["p_value",""]]


reject_null,False,True
gt_null,,
False,196,3509
True,80,40
